In [2]:
import pandas as pd
import statsmodels.api as sm


In [3]:
df = pd.read_csv("../data/processed/features.csv")
df.head()


,date,home_team,away_team,home_goals,away_goals,home_avg_scored,home_avg_conceded,away_avg_scored,away_avg_conceded,home_advantage
0,2020-09-20,Parma,Napoli,0.0,2.0,3.0,2.4,2.8,0.4,1
1,2020-09-21,Milan,Bologna,2.0,0.0,2.6,2.2,2.4,1.0,1
2,2020-09-26,Torino,Atalanta,2.0,4.0,2.0,2.2,1.6,0.8,1
3,2020-09-26,Cagliari,Lazio,0.0,2.0,1.0,1.8,1.2,1.8,1
4,2020-09-26,Sampdoria,Benevento,2.0,3.0,0.8,1.4,1.4,2.0,1


In [4]:
df["date"] = pd.to_datetime(df["date"])

def season_from_date(d):
    return d.year + 1 if d.month >= 7 else d.year

df["season"] = df["date"].apply(season_from_date)

df["season"].value_counts().sort_index()


season
2021    331
2022    321
2023    337
2024    336
2025    341
Name: count, dtype: int64

In [5]:
# Drop incomplete 2024/25 season
df = df[df["season"] <= 2024].copy()

df["season"].value_counts().sort_index()


season
2021    331
2022    321
2023    337
2024    336
Name: count, dtype: int64

In [6]:
train = df[df["season"] < 2024]
test = df[df["season"] == 2024]

train.shape, test.shape


((989, 11), (336, 11))

In [7]:
FEATURES = [
    "home_avg_scored",
    "home_avg_conceded",
    "away_avg_scored",
    "away_avg_conceded",
    "home_advantage",
]

X_train = sm.add_constant(train[FEATURES])
X_train.head()

,home_avg_scored,home_avg_conceded,away_avg_scored,away_avg_conceded,home_advantage
0,3.0,2.4,2.8,0.4,1
1,2.6,2.2,2.4,1.0,1
2,2.0,2.2,1.6,0.8,1
3,1.0,1.8,1.2,1.8,1
4,0.8,1.4,1.4,2.0,1


In [8]:
home_model = sm.GLM(
    train["home_goals"],
    X_train,
    family=sm.families.Poisson()
).fit()

home_model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:             home_goals   No. Observations:                  989
Model:                            GLM   Df Residuals:                      984
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1520.2
Date:                Wed, 24 Dec 2025   Deviance:                       1174.1
Time:                        17:13:46   Pearson chi2:                 1.04e+03
No. Iterations:                     5   Pseudo R-squ. (CS):           0.002818
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
home_avg_scored       0.0544      0.041      1.315      0.189      -0.027       0.135
home_avg_conceded    -0.0367      0.044     -0.830      0.407      -0.123       0.050
away_avg_scored      -0.0017      0.043     -0.040      0.968      -0.087       0.083
away_avg_conceded    -0.0071      0.048     -0.147      0.883      -0.101       0.087
home_advantage        0.3717      0.135      2.745      0.006       0.106       0.637
=====================================================================================
"""

In [9]:
away_model = sm.GLM(
    train["away_goals"],
    X_train,
    family=sm.families.Poisson()
).fit()

away_model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:             away_goals   No. Observations:                  989
Model:                            GLM   Df Residuals:                      984
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1461.7
Date:                Wed, 24 Dec 2025   Deviance:                       1154.2
Time:                        17:15:24   Pearson chi2:                     985.
No. Iterations:                     5   Pseudo R-squ. (CS):           0.003414
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
home_avg_scored      -0.0006      0.044     -0.015      0.988      -0.086       0.085
home_avg_conceded    -0.0642      0.046     -1.387      0.166      -0.155       0.027
away_avg_scored       0.0552      0.045      1.230      0.219      -0.033       0.143
away_avg_conceded     0.0187      0.050      0.375      0.708      -0.079       0.117
home_advantage        0.2881      0.141      2.042      0.041       0.012       0.565
=====================================================================================
"""